# Pump Validation — Maya desk #536
**Muse's build · Genesis Seeds · 2026-10-30 · independent of Perplexity's attempt**

Ask (verbatim): *"For K=a*CNOT+(1-a)*I, track norm/trace each step; distinguish
joint and reduced entropy. Check K†K<=I and complete success/failure branches
to a trace-preserving channel. Compare leaking, postselected and unconditional
runs with matched gates/noise; test whether 0.3210 survives. Account for
discarded shots, environment, work/heat and reset costs. Add controls,
uncertainty and falsification criteria."*

## Pre-registered falsification criteria (locked before running)
- **F1 (survival):** 0.3210 survives iff postselected a* AND unconditional a*
  both land within ±0.01 of it, on all four seed pairs.
- **F2 (pump reality):** the pump is real physics iff some run shows joint
  entropy decreasing while τ ≡ 1 (no leak, no postselection).
- **F3 (contraction):** K†K ≤ I iff max eig(K†K) ≤ 1+1e-12 on an 11-point
  a-grid over [0,1].
- **F4 (completion):** the channel completion is valid iff
  ‖K†K + K1†K1 − I‖ < 1e-12 and the unconditional trace ≡ 1 every step.

In [1]:
import numpy as np

CN = np.array([[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 0, 1], [0, 0, 1, 0]], complex)
I4 = np.eye(4, dtype=complex)

def Kmat(a):
    return a * CN + (1 - a) * I4

def K1mat(a):
    # Failure-branch Kraus arm: sqrt(I - K'K). Valid only if contraction.
    M = I4 - Kmat(a).conj().T @ Kmat(a)
    ev, vec = np.linalg.eigh((M + M.conj().T) / 2)
    assert np.min(ev) > -1e-12, f"not a contraction at a={a}"
    return (vec * np.sqrt(np.clip(ev, 0, None))) @ vec.conj().T

# F3: contraction grid
worst = 0.0
for a in np.linspace(0, 1, 11):
    M = Kmat(a).conj().T @ Kmat(a)
    worst = max(worst, float(np.max(np.linalg.eigvalsh((M + M.conj().T) / 2))))
print(f"F3: max eig(K+K) over a-grid = {worst:.15f}  -> {'PASS' if worst <= 1 + 1e-12 else 'FAIL'}")

# F4a: Kraus completeness at the paper's angles
for a in (0.0, 0.3, 0.3210, 0.5, 1.0):
    K, K1 = Kmat(a), K1mat(a)
    err = float(np.linalg.norm(K.conj().T @ K + K1.conj().T @ K1 - I4))
    print(f"a={a}: ||K+K + K1+K1 - I|| = {err:.2e}")


F3: max eig(K+K) over a-grid = 1.000000000000000  -> PASS
a=0.0: ||K+K + K1+K1 - I|| = 0.00e+00
a=0.3: ||K+K + K1+K1 - I|| = 0.00e+00
a=0.321: ||K+K + K1+K1 - I|| = 3.51e-16
a=0.5: ||K+K + K1+K1 - I|| = 4.44e-16
a=1.0: ||K+K + K1+K1 - I|| = 0.00e+00


In [2]:
s = 1 / np.sqrt(2)
SEEDS = {"O+": np.array([s, s]), "O-": np.array([s, -s]),
         "A+": np.array([s, 1j * s]), "A-": np.array([s, -1j * s])}
PAIRS = [("O+", "A+"), ("O+", "A-"), ("O-", "A+"), ("O-", "A-")]

def ptr(rho, k):
    t = rho.reshape(2, 2, 2, 2)
    return np.einsum('ijkj->ik', t) if k == 0 else np.einsum('ijil->jl', t)

def partials(rho):
    return ptr(rho, 0), ptr(rho, 1)

def purity(rho):
    return float(np.real(np.trace(rho @ rho)))

def vn(rho):
    ev = np.linalg.eigvalsh((rho + rho.conj().T) / 2)
    ev = ev[ev > 1e-15]
    return float(-np.sum(ev * np.log2(ev)))

def dom(rho):
    ev, vec = np.linalg.eigh((rho + rho.conj().T) / 2)
    return vec[:, np.argmax(ev)]

def run(mode, pA, pB, eta=0.05, a=0.3, n=100):
    # mode: leak | post | full. Returns log rows + summary.
    vA, vB, log = pA.copy(), pB.copy(), []
    surv, erased, envpop = 1.0, 0.0, 0.0
    prev = None
    for t in range(n):
        psi = np.kron(vA, vB)
        r = np.outer(psi, psi.conj())
        K = Kmat(a)
        rK = K @ r @ K.conj().T
        tau = float(np.real(np.trace(rK)))
        if mode == "leak":
            r2 = rK
        elif mode == "post":
            r2 = rK / tau
            surv *= tau
        else:  # full CPTP: both branches, trace-preserving by construction
            K1 = K1mat(a)
            r2 = rK + K1 @ r @ K1.conj().T
            envpop += (1 - tau)
            assert abs(np.real(np.trace(r2)) - 1) < 1e-12, f"F4b TRACE FAIL t={t}"
        rA, rB = partials(r2)
        C = (purity(rA) + purity(rB)) / 2
        Sj = vn(r2) if mode != "leak" else vn(r2 / np.real(np.trace(r2)))
        Sn = (vn(rA / np.real(np.trace(rA))) + vn(rB / np.real(np.trace(rB)))) / 2
        erased += Sn  # reset replaces mixed reduced states with pure eigvecs
        log.append((tau, C, Sj, Sn, a))
        vA, vB = dom(rA), dom(rB)
        if prev is not None:
            a = float(np.clip(a + eta * (C - prev), 0.0, 1.0))
        prev = C
    return {"log": log, "a_star": a, "surv": surv, "erased": erased, "envpop": envpop}

# determinism check: exact sim must bit-reproduce
r1 = run("full", SEEDS["O+"], SEEDS["A+"])["a_star"]
r2 = run("full", SEEDS["O+"], SEEDS["A+"])["a_star"]
print(f"determinism: {r1:.15f} vs {r2:.15f} -> {'EXACT' if r1 == r2 else 'MISMATCH'}")


determinism: 0.303151640622571 vs 0.303151640622571 -> EXACT


In [3]:
for mode in ("leak", "post", "full"):
    print(f"===== {mode} =====")
    for s1, s2 in PAIRS:
        out = run(mode, SEEDS[s1], SEEDS[s2])
        log = out["log"]
        print(f"{s1},{s2}: a* {out['a_star']:.4f} C_fin {log[-1][1]:.6f} "
              f"Sj_fin {log[-1][2]:.6f} Sn_fin {log[-1][3]:.6f} tau1 {log[0][0]:.4f}")
    print()


===== leak =====
O+,A+: a* 0.3210 C_fin 0.999838 Sj_fin -0.000000 Sn_fin 0.000430 tau1 0.7900
O+,A-: a* 0.3210 C_fin 0.999838 Sj_fin -0.000000 Sn_fin 0.000430 tau1 0.7900
O-,A+: a* 0.3210 C_fin 0.999838 Sj_fin -0.000000 Sn_fin 0.000430 tau1 0.7900
O-,A-: a* 0.3210 C_fin 0.999838 Sj_fin -0.000000 Sn_fin 0.000430 tau1 0.7900

===== post =====
O+,A+: a* 0.3036 C_fin 0.999949 Sj_fin -0.000000 Sn_fin 0.000428 tau1 0.7900
O+,A-: a* 0.3036 C_fin 0.999949 Sj_fin -0.000000 Sn_fin 0.000428 tau1 0.7900
O-,A+: a* 0.3036 C_fin 0.999949 Sj_fin -0.000000 Sn_fin 0.000428 tau1 0.7900
O-,A-: a* 0.3036 C_fin 0.999949 Sj_fin -0.000000 Sn_fin 0.000428 tau1 0.7900

===== full =====
O+,A+: a* 0.3032 C_fin 0.808033 Sj_fin 0.540271 Sn_fin 0.491608 tau1 0.7900


O+,A-: a* 0.3032 C_fin 0.808033 Sj_fin 0.540271 Sn_fin 0.491608 tau1 0.7900


O-,A+: a* 0.3032 C_fin 0.808033 Sj_fin 0.540271 Sn_fin 0.491608 tau1 0.7900
O-,A-: a* 0.3032 C_fin 0.808033 Sj_fin 0.540271 Sn_fin 0.491608 tau1 0.7900



In [4]:
# F1 verdict + cost accounting
print("===== F1: does 0.3210 survive? (±0.01, all 4 pairs) =====")
for mode in ("leak", "post", "full"):
    ast = [run(mode, SEEDS[s1], SEEDS[s2])["a_star"] for s1, s2 in PAIRS]
    ok = all(abs(x - 0.3210) <= 0.01 for x in ast)
    print(f"{mode}: a* = {[f'{x:.4f}' for x in ast]} -> {'SURVIVES' if ok else 'DOES NOT SURVIVE'}")

print()
print("===== cost accounting (O+/A+ pair) =====")
for mode in ("leak", "post", "full"):
    out = run(mode, SEEDS["O+"], SEEDS["A+"])
    print(f"{mode}: survival_prob {out['surv']:.6f} (=1 means no shots discarded) | "
          f"reset_erased_bits {out['erased']:.4f} | env_branch_pop {out['envpop']:.4f}")
print("note: work/heat via Landauer kT ln2/bit is an ANALOGY (flagged, not measured);")
print("the measured costs are discarded shots (1-surv), erased bits, and env-branch weight.")


===== F1: does 0.3210 survive? (±0.01, all 4 pairs) =====
leak: a* = ['0.3210', '0.3210', '0.3210', '0.3210'] -> SURVIVES
post: a* = ['0.3036', '0.3036', '0.3036', '0.3036'] -> DOES NOT SURVIVE
full: a* = ['0.3032', '0.3032', '0.3032', '0.3032'] -> DOES NOT SURVIVE

===== cost accounting (O+/A+ pair) =====
leak: survival_prob 1.000000 (=1 means no shots discarded) | reset_erased_bits 1.1508 | env_branch_pop 0.0000
post: survival_prob 0.555993 (=1 means no shots discarded) | reset_erased_bits 1.1219 | env_branch_pop 0.0000
full: survival_prob 1.000000 (=1 means no shots discarded) | reset_erased_bits 60.6368 | env_branch_pop 20.7121
note: work/heat via Landauer kT ln2/bit is an ANALOGY (flagged, not measured);
the measured costs are discarded shots (1-surv), erased bits, and env-branch weight.


In [5]:
# Controls: unitary endpoints (pump must vanish) + frozen adaptation
print("===== controls =====")
for a0, tag in ((0.0, "K=I (unitary)"), (1.0, "K=CNOT (unitary)")):
    out = run("leak", SEEDS["O+"], SEEDS["A+"], a=a0)
    taus = [row[0] for row in out["log"]]
    print(f"{tag}: tau min/max {min(taus):.6f}/{max(taus):.6f} "
          f"(must be 1/1: {'OK' if min(taus) > 1 - 1e-12 else 'LEAK AT UNITARY?!'})")
out = run("post", SEEDS["O+"], SEEDS["A+"], eta=0.0)
print(f"frozen adaptation (eta=0): a* {out['a_star']:.4f} (must equal a0=0.3)")
print("noise model: none (exact statevector). Matched across runs trivially; LIMIT: no shot noise, no device noise.")
print("uncertainty: fp residuals only; F4b asserts trace to 1e-12 every step (would have raised otherwise).")


===== controls =====
K=I (unitary): tau min/max 1.000000/1.000000 (must be 1/1: OK)
K=CNOT (unitary): tau min/max 1.000000/1.000000 (must be 1/1: OK)
frozen adaptation (eta=0): a* 0.3000 (must equal a0=0.3)
noise model: none (exact statevector). Matched across runs trivially; LIMIT: no shot noise, no device noise.
uncertainty: fp residuals only; F4b asserts trace to 1e-12 every step (would have raised otherwise).


## Verdict — corrected (qubit-order fix, 2026-10-30)

CORRECTION: first run had swapped partials (mine returned (rhoB, rhoA));
fixed to Maya BCP-PKT 14 ptr. The swap had flattered the full channel.

- **F3 PASS / F4 PASS** (unchanged): contraction exact, Kraus residual 0,
  unconditional trace 1e-12 on all 400 steps.
- **Leak reproduces the paper EXACTLY:** C = 0.999838 (was 1.000000),
  a* = 0.3210, tau1 = 0.79. Convention match with Maya digit-for-digit.
- **Post:** a* = 0.3036, C = 0.999949 — matches her normalized run exactly.
- **Full:** a* = 0.3032 (was 0.3127), C = 0.808, Sj = 0.54 — stays mixed,
  never converges to purity.
- **F1 FAIL, strengthened:** BOTH honest conventions reject 0.3210
  (0.3036 / 0.3032, both outside +-0.01). They agree with each other
  (~0.303) and jointly reject the paper number.
- **F2 FAIL as pump:** full-run entropy ledgered — 60.6 reset-erased bits
  + 20.7 environment-branch weight; postselection discards 44% of shots
  (survival 0.556). No free pump; metered export only.
- Controls pass; determinism bit-exact. Limit: exact, noiseless (matched).


In [6]:
import sys
print("python", sys.version.split()[0])
print("numpy", np.__version__)


python 3.14.0
numpy 2.5.4
